In [ ]:
import extraccion.minioFunctions as mf
import pandas as pd
anios = [2022, 2023, 2024, 2025]
dfgen = []
dfs = []
for k in anios:
    dfs2 = mf.bajar_fichero(mf.crear_cliente(), path_server=f'grupo3/raw/Suelo2/Suelo2_{k}.parquet', type= 'df')
    dfg = mf.bajar_fichero(mf.crear_cliente(), path_server=f'grupo3/raw/Final/final_{k}.parquet', type= 'df' )
    dfs.append(dfs2)    
    dfgen.append(dfg)

final = pd.concat(dfgen, ignore_index=True)
suelo2 = pd.concat(dfs, ignore_index=True)

In [ ]:
final.info()

In [ ]:
suelo2.info()

In [ ]:
aniadir = mf.bajar_fichero(mf.crear_cliente(), path_server='grupo3/raw/Suelo2/sinNulos_general.parquet',type='df')
aniadir.sort_values(by='date')

In [ ]:
df_merged = pd.merge(suelo2, final, on=['lat', 'lon'], how='right', indicator=True)
# print(len(df_merged))
df_diferencia = df_merged[df_merged['_merge'] != 'both'].copy()
df_diferencia = df_diferencia.drop(columns=['fire_index', '_merge', 'date_x'])

In [29]:
suelo2_unico = suelo2.drop_duplicates(subset=['lat', 'lon'], keep='first')
df_merged = pd.merge(final, suelo2_unico, on=['lat', 'lon'], how='left')

df_merged = df_merged.set_index(['lat', 'lon'])
aniadir_idx = aniadir.set_index(['lat', 'lon'])

df_merged['soil_temp'] = df_merged['soil_temp'].fillna(aniadir_idx['soil_temp'])

df_final = df_merged.reset_index()

In [30]:
df_final = df_final.rename(columns={'date_x':'date'}).sort_values(by='date').reset_index().drop(columns=['date_y', 'index', 'fire_index'])

In [31]:
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 71444 entries, 0 to 71443
Data columns (total 22 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   lat                 71444 non-null  float64       
 1   lon                 71444 non-null  float64       
 2   date                71444 non-null  datetime64[us]
 3   final               71444 non-null  int64         
 4   elevacion_centro    71444 non-null  float64       
 5   grados              71444 non-null  float64       
 6   porcentaje          71444 non-null  float64       
 7   temp_mean           71444 non-null  float64       
 8   temp_max            71444 non-null  float64       
 9   temp_min            71444 non-null  float64       
 10  humidity_mean       71444 non-null  float64       
 11  precipitation       71444 non-null  float64       
 12  wind_speed_max      71444 non-null  float64       
 13  wind_gusts_max      71444 non-null  float64       
 14  p

In [33]:
dfs_por_anio = {
    int(anio): grupo.copy() 
    for anio, grupo in df_final.dropna(subset=['date']).groupby(df_final['date'].dt.year)
}

In [35]:
for anio, df_anio in dfs_por_anio.items():
    mf.subir_fichero(mf.crear_cliente(), path_server=f'grupo3/raw/Final/final_{anio}.parquet', df=df_anio)

Fichero subido como grupo3/raw/Final/final_2022.parquet
Fichero subido como grupo3/raw/Final/final_2023.parquet
Fichero subido como grupo3/raw/Final/final_2024.parquet
Fichero subido como grupo3/raw/Final/final_2025.parquet


In [ ]:
'''
RENOMBRAR COLUMNAS PARA COMPATIBILIDAD
df = mf.bajar_fichero(mf.crear_cliente(), path_server='grupo3/raw/Incendios_y_no_incendios/suelo_temporal.parquet', type='df')
df = df.rename(columns={'lat_mean':'lat', 'lon_mean':'lon'})
mf.subir_fichero(mf.crear_cliente(), path_server='grupo3/raw/Incendios_y_no_incendios/suelo_temporal.parquet', df=df)
'''